# Radio Signal Parameter Estimation: Issues & Solutions Log

This notebook documents the process of training a physics-informed neural network to estimate the phase offset of a 16-QAM radio signal and reconstruct it. Below is a summary of the critical issues encountered during development and the solutions that led to a successful model (BER < 1%).

### 1. The "Symmetry Ambiguity" Problem

* **Issue:** The initial model converged to a loss of ~1.0 (random guessing) and a BER of ~0.85.
* **Root Cause:** 16-QAM is rotationally symmetric every $90^\circ$ ($\pi/2$). Without an external reference, the network cannot distinguish between true $\theta$ and $\theta+90^\circ$. It attempts to average these possibilities, leading to a destructive mean of 0.
* **Solution:** **Pilot-Aided Estimation**. We explicitly feed the first $N=4$ known symbols ("pilots") into the network. These act as a "phase anchor," breaking the symmetry and allowing the network to lock onto the correct quadrant.

### 2. The "Lazy Network" / Vanishing Gradient Problem

* **Issue:** Even with pilots, the network initially ignored them because the sparse pilot signal ($8$ floats) was drowned out by the noisy data signal ($1024$ floats).
* **Root Cause:** The network struggled to learn the specific complex-conjugate arithmetic required to extract phase from pilots from scratch (the optimization landscape was too flat).
* **Solution:** **Physics-Informed Residual Learning**. We calculate a "Classical Hint" (a rough Coarse Estimate using the pilots via Least Squares) and feed this vector $[\cos \hat{\theta}, \sin \hat{\theta}]$ into the network.
* *Result:* The NN no longer needs to learn the phase from zero; it only needs to learn the **Residual Correction** ($\Delta \theta$) to account for noise and non-linearities, which is a much easier task.



### 3. Input Normalization & SNR Curriculum

* **Issue:** Gradients were unstable, and the MLP struggled to converge on raw IQ data.
* **Solution:**
1. **Batch Normalization:** Added `BatchNorm1d` immediately after flattening inputs to center the data and keep variance unit-scale.
2. **SNR Randomization:** We trained on a dynamic SNR range ($15 - 30$ dB). This prevented the model from overfitting to clean signals and forced it to learn robust feature extraction for noisy environments.



### 4. Manifold-Aware Loss

* **Issue:** Using MSE (Mean Squared Error) on raw angles fails because $-\pi$ and $+\pi$ are far apart in Euclidean numbers but identical in physical phase (the "wrap-around" problem).
* **Solution:** We predict a vector $[\cos \theta, \sin \theta]$ and maximize the cosine similarity. The loss function is $L = 1 - \cos(\theta_{pred} - \theta_{true})$, which is differentiable and correctly respects the circular geometry of the phase manifold.

---

In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import matplotlib.pyplot as plt
import random
import time

# -------------------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 50 
WARMUP_EPOCHS = 10 # Train ONLY physics for first 10 epochs
LR = 0.001
SEQ_LEN = 256
N_PILOTS = 4
DATA_PATH = "./data/radio_divina_commedia.pth"

class Colors:
    GREEN = '\033[92m'; RED = '\033[91m'; RESET = '\033[0m'; BOLD = '\033[1m'

print(f"Using device: {DEVICE}")

# -------------------------------------------------------------------------------------
# 2. Helpers
# -------------------------------------------------------------------------------------
def build_verified_constellation(path):
    ckpt = torch.load(path, map_location=DEVICE)
    iq_c = ckpt['iq_clean']
    z_c = torch.complex(iq_c[0,0,:], iq_c[0,1,:])
    z_np = z_c.cpu().numpy()
    const_np = np.unique(z_np) # Numpy handles complex unique correctly
    
    if len(const_np) < 16: 
        b = torch.tensor([-3.,-1.,1.,3.], device=DEVICE)
        g = torch.meshgrid(b, b, indexing='ij')
        const = (g[0] + 1j*g[1]).flatten()
        const = const / torch.sqrt(torch.mean(torch.abs(const)**2))
        return const
    return torch.from_numpy(const_np).to(DEVICE)

CONST_TENSOR = build_verified_constellation(DATA_PATH)

def classical_pilot_estimate(noisy_pilots, clean_pilots):
    phasor = (noisy_pilots * torch.conj(clean_pilots)).sum(dim=1)
    return torch.stack([phasor.real, phasor.imag], dim=1).float()

def demap_16qam(iq_tensor):
    dist = torch.abs(iq_tensor.unsqueeze(-1) - CONST_TENSOR.view(1, 1, -1))
    return torch.argmin(dist, dim=-1)

def calc_ber(pred_idx, true_idx):
    diff = pred_idx ^ true_idx
    b0 = (diff & 1); b1 = ((diff >> 1) & 1); b2 = ((diff >> 2) & 1); b3 = ((diff >> 3) & 1)
    return (b0 + b1 + b2 + b3).sum().float() / (pred_idx.numel() * 4)

def decode_text(indices, mask):
    try:
        ints = indices.cpu().numpy().astype(np.uint8)
        bits = np.unpackbits(ints[:, None], axis=1)[:, -4:].flatten()
        m = mask.cpu().numpy().flatten()[:len(bits)]
        clean = np.bitwise_xor(bits, m)
        return np.packbits(clean).tobytes().replace(b'\x00', b'').decode('utf-8', 'ignore')
    except: return "."

# -------------------------------------------------------------------------------------
# 3. Dataset
# -------------------------------------------------------------------------------------
def load_and_split_data(path, batch_size=BATCH_SIZE, seed=42):
    global SEQ_LEN
    print(f"Loading {path}...")
    if not os.path.exists(path): raise FileNotFoundError("Dataset not found")
    
    ckpt = torch.load(path, map_location=DEVICE)
    x = ckpt['iq_noisy'].float()
    y_phi = ckpt['phase_labels'].float().reshape(-1)
    y_cfo = ckpt['cfo_labels'].float().reshape(-1)
    bits = ckpt['bits']
    masks = ckpt.get('scramble_mask', torch.zeros_like(bits))

    if x.shape[2] != SEQ_LEN: SEQ_LEN = x.shape[2]

    b_re = bits.reshape(bits.shape[0], -1, 4).long()
    z = (b_re[:,:,0]*8 + b_re[:,:,1]*4 + b_re[:,:,2]*2 + b_re[:,:,3]*1)
    
    ds = TensorDataset(x, y_phi, y_cfo, z, masks)
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    
    train_ds = Subset(ds, idx[:int(0.75*len(ds))])
    val_ds   = Subset(ds, idx[int(0.75*len(ds)):int(0.95*len(ds))])
    test_ds  = Subset(ds, idx[int(0.95*len(ds)):])
    
    print(f"Split: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")
    return (DataLoader(train_ds, batch_size, shuffle=True), 
            DataLoader(val_ds, batch_size, False), 
            DataLoader(test_ds, batch_size, False))

train_loader, val_loader, test_loader = load_and_split_data(DATA_PATH)

# -------------------------------------------------------------------------------------
# 4. Neural Receiver
# -------------------------------------------------------------------------------------
class ResidualBlock1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

class NeuralReceiver(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, n_pilots=N_PILOTS):
        super().__init__()
        
        # ESTIMATOR
        in_dim = (2 * seq_len) + (2 * n_pilots) + 2
        self.input_norm = nn.BatchNorm1d(in_dim)
        self.estimator = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.BatchNorm1d(512),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Linear(256, 3) 
        )
        
        # REFINER
        self.refiner = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
            ResidualBlock1D(64), ResidualBlock1D(64),
            nn.Conv1d(64, 16, 1) 
        )

    def forward(self, x, pilots, hint):
        # 1. Estimate
        x_flat = x.view(x.size(0), -1)
        p_flat = pilots.view(pilots.size(0), -1)
        feats = self.input_norm(torch.cat([x_flat, p_flat, hint], dim=1))
        
        params = self.estimator(feats)
        phi = torch.atan2(params[:, 1], params[:, 0])
        cfo = params[:, 2] / 100.0
        
        # 2. Correct Physics
        B, _, L = x.shape
        t = torch.arange(L, device=x.device).float().unsqueeze(0)
        phase = phi.unsqueeze(1) + (cfo.unsqueeze(1) * t)
        
        cos_t, sin_t = torch.cos(phase).unsqueeze(1), torch.sin(phase).unsqueeze(1)
        r_I = x[:,0,:] * cos_t.squeeze(1) + x[:,1,:] * sin_t.squeeze(1)
        r_Q = -x[:,0,:] * sin_t.squeeze(1) + x[:,1,:] * cos_t.squeeze(1)
        x_coarse = torch.stack([r_I, r_Q], dim=1)
        
        # 3. Refine
        logits = self.refiner(x_coarse)
        return logits, phi, cfo

# -------------------------------------------------------------------------------------
# 5. Training with WARMUP
# -------------------------------------------------------------------------------------
model = NeuralReceiver(SEQ_LEN, N_PILOTS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
loss_ce = nn.CrossEntropyLoss()
loss_mse = nn.MSELoss()

print(f"--- Starting Training ({EPOCHS} Epochs, Warmup={WARMUP_EPOCHS}) ---")

for epoch in range(EPOCHS):
    model.train()
    metrics = {'loss':0, 'cls':0, 'phi':0, 'cfo':0, 'ber':0}
    
    # Dynamic Weighting: Phase 1 (Warmup) vs Phase 2 (End-to-End)
    if epoch < WARMUP_EPOCHS:
        w_cls, w_phys = 0.0, 1.0
        mode = "WARMUP (Physics Only)"
    else:
        w_cls, w_phys = 1.0, 0.1
        mode = "E2E (Symbol + Physics)"

    for x, y_p, y_c, z, _ in train_loader:
        x, y_p, y_c, z = x.to(DEVICE), y_p.to(DEVICE), y_c.to(DEVICE), z.to(DEVICE)
        
        p_ref = CONST_TENSOR[z[:, :N_PILOTS]]
        x_c = torch.complex(x[:,0], x[:,1])
        hint = classical_pilot_estimate(x_c[:, :N_PILOTS], p_ref)
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        
        logits, phi, cfo = model(x, p_feat, hint)
        
        # Losses
        l_cls = loss_ce(logits, z)
        # Physics Loss: Phase + Scaled CFO
        l_phys = (1.0 - torch.cos(phi - y_p).mean()) + loss_mse(cfo*100, y_c*100)
        
        loss = (w_cls * l_cls) + (w_phys * l_phys)
        
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        
        # Track
        with torch.no_grad():
            metrics['loss'] += loss.item()
            metrics['cls'] += l_cls.item()
            metrics['phi'] += torch.abs(torch.atan2(torch.sin(phi-y_p), torch.cos(phi-y_p))).mean().item()
            metrics['cfo'] += torch.abs(cfo - y_c).mean().item()
            metrics['ber'] += calc_ber(torch.argmax(logits,1), z).item()

    # Val
    model.eval()
    val_ber = 0
    with torch.no_grad():
        for x, _, _, z, _ in val_loader:
            x, z = x.to(DEVICE), z.to(DEVICE)
            p_ref = CONST_TENSOR[z[:, :N_PILOTS]]
            hint = classical_pilot_estimate(torch.complex(x[:,0], x[:,1])[:, :N_PILOTS], p_ref)
            p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
            
            logits, _, _ = model(x, p_feat, hint)
            val_ber += calc_ber(torch.argmax(logits,1), z).item() * z.numel()*4
            
    n = len(train_loader)
    print(f"Ep {epoch+1:02d} [{mode}] | Loss: {metrics['loss']/n:.3f} | "
          f"Phi Err: {metrics['phi']/n:.3f} rad | CFO Err: {metrics['cfo']/n:.5f} | "
          f"Val BER: {val_ber/(len(val_loader.dataset)*SEQ_LEN*4):.5f}")

MODEL_PATH = "./models/Neural_Receiver.pth"
torch.save(model.state_dict(), MODEL_PATH)

# -------------------------------------------------------------------------------------
# 6. Benchmark
# -------------------------------------------------------------------------------------
print(f"\n{Colors.BOLD}{'ID':<3} | {'Method':<10} | {'Phase Err':<10} | {'BER':<8} | {'Latency':<8} | {'Text Reconstruction'}{Colors.RESET}")
print("-" * 150)

model.eval()
res = {'c_ber':[], 'n_ber':[], 'c_tm':[], 'n_tm':[]}
cnt = 0

with torch.no_grad():
    for x, y_p, y_c, z, m in test_loader:
        x, z, m = x.to(DEVICE), z.to(DEVICE), m.to(DEVICE)
        y_p = y_p.to(DEVICE)
        x_c = torch.complex(x[:,0], x[:,1])
        
        # CLASSICAL
        t0 = time.perf_counter()
        p_ref = CONST_TENSOR[z[:, :N_PILOTS]]
        theta = torch.angle((x_c[:, :N_PILOTS] * torch.conj(p_ref)).sum(dim=1))
        rx_cl = x_c * torch.exp(-1j * theta.unsqueeze(1))
        idx_cl = demap_16qam(rx_cl)
        t_cl = (time.perf_counter() - t0)*1000/x.size(0)
        
        # NEURAL
        t0 = time.perf_counter()
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        hint = torch.stack([(x_c[:,:N_PILOTS]*torch.conj(p_ref)).sum(1).real, (x_c[:,:N_PILOTS]*torch.conj(p_ref)).sum(1).imag],1).float()
        logits, phi_n, _ = model(x, p_feat, hint)
        idx_nn = torch.argmax(logits, dim=1)
        t_nn = (time.perf_counter() - t0)*1000/x.size(0)
        
        for i in range(x.size(0)):
            res['c_ber'].append(calc_ber(idx_cl[i], z[i]).item())
            res['n_ber'].append(calc_ber(idx_nn[i], z[i]).item())
            res['c_tm'].append(t_cl); res['n_tm'].append(t_nn)
            
            if cnt < 5:
                pay = slice(N_PILOTS, None)
                truth = decode_text(z[i, pay], m[i, N_PILOTS*4:])
                txt_c = decode_text(idx_cl[i, pay], m[i, N_PILOTS*4:])
                txt_n = decode_text(idx_nn[i, pay], m[i, N_PILOTS*4:])
                pe_c = abs(theta[i].item() - y_p[i].item())
                pe_n = abs(phi_n[i].item() - y_p[i].item())
                
                def hl(t, p): return f"{Colors.GREEN}{p}{Colors.RESET}" if t==p else f"{t[:15]}.. vs {Colors.RED}{p[:15]}..{Colors.RESET}"
                print(f"{cnt+1:<3} | Classical  | {pe_c:.4f} rad | {res['c_ber'][-1]:.4f}   | {t_cl:.4f}ms | {hl(truth, txt_c)}")
                print(f"{'':<3} | Neural Net | {pe_n:.4f} rad | {res['n_ber'][-1]:.4f}   | {t_nn:.4f}ms | {hl(truth, txt_n)}")
                print("-" * 150)
                cnt += 1

print(f"Classical  -> BER: {np.mean(res['c_ber']):.5f}")
print(f"Neural Net -> BER: {np.mean(res['n_ber']):.5f}")

Using device: cpu
Loading ./data/radio_divina_commedia.pth...
Split: Train=3855, Val=1028, Test=257
--- Starting Training (50 Epochs, Warmup=10) ---
Ep 01 [WARMUP (Physics Only)] | Loss: 0.939 | Phi Err: 1.343 rad | CFO Err: 0.00260 | Val BER: 0.49528
Ep 02 [WARMUP (Physics Only)] | Loss: 0.486 | Phi Err: 0.879 rad | CFO Err: 0.00115 | Val BER: 0.49228
Ep 03 [WARMUP (Physics Only)] | Loss: 0.298 | Phi Err: 0.640 rad | CFO Err: 0.00072 | Val BER: 0.49158
Ep 04 [WARMUP (Physics Only)] | Loss: 0.241 | Phi Err: 0.564 rad | CFO Err: 0.00059 | Val BER: 0.49065
Ep 05 [WARMUP (Physics Only)] | Loss: 0.164 | Phi Err: 0.446 rad | CFO Err: 0.00051 | Val BER: 0.49055
Ep 06 [WARMUP (Physics Only)] | Loss: 0.120 | Phi Err: 0.372 rad | CFO Err: 0.00047 | Val BER: 0.49057
Ep 07 [WARMUP (Physics Only)] | Loss: 0.088 | Phi Err: 0.311 rad | CFO Err: 0.00044 | Val BER: 0.49051
Ep 08 [WARMUP (Physics Only)] | Loss: 0.066 | Phi Err: 0.267 rad | CFO Err: 0.00043 | Val BER: 0.49016
Ep 09 [WARMUP (Physics Only